# 02 Web后端开发 · 第01课：HTTP与API基础

这一课默认你以前没有系统学习过网站后端。我们先不使用FastAPI，而是把网页与后端通信时最核心的概念讲清楚。

## 本课目标

完成本课后，你应该能够：

1. 区分客户端、服务器、前端和后端；
2. 看懂一个URL的组成；
3. 理解HTTP请求和HTTP响应；
4. 知道GET、POST、PUT、PATCH、DELETE通常分别表示什么；
5. 理解状态码、请求头、请求体和JSON；
6. 看懂简单的API说明；
7. 用Python字典模拟一个简单API。

## 1. 一次网页访问发生了什么？

你在浏览器输入网址并按下回车时，可以先把过程理解成：

```text
浏览器（客户端） ──发送请求──> 服务器（后端）
浏览器（客户端） <──返回响应── 服务器（后端）
```

- **客户端（Client）**：主动发起请求的一方，例如浏览器、手机App或Python程序。
- **服务器（Server）**：接收请求、执行处理并返回结果的一方。
- **前端（Frontend）**：用户能看到和操作的界面。
- **后端（Backend）**：处理业务、调用模型、访问数据库并返回数据的程序。

客户端和服务器描述的是通信角色；前端和后端描述的是系统分工。浏览器通常是客户端，但Python脚本也可以成为客户端。

## 2. HTTP是什么？

HTTP是客户端和服务器之间的一套通信规则。双方按照相同格式表达“我想做什么”和“处理结果是什么”。

一次HTTP通信总是包含两部分：

1. **Request（请求）**：客户端发给服务器；
2. **Response（响应）**：服务器处理后返回给客户端。

请求与响应不是同一个东西。客户端不能只说“给我数据”，还需要说明访问地址、请求方法，必要时还要携带数据。

## 3. URL的组成

观察这个地址：

```text
https://api.example.com:443/chats/12/messages?limit=10&order=desc
```

- `https`：协议；
- `api.example.com`：主机名，表示要访问哪台服务器；
- `443`：端口，HTTPS默认使用443，因此经常省略；
- `/chats/12/messages`：路径，表示要访问的资源；
- `?limit=10&order=desc`：查询参数，用来附加筛选或控制条件。

路径参数通常写在路径中，例如`12`可能是聊天ID；查询参数位于`?`后，多组参数使用`&`连接。

In [ ]:
# urllib.parse是Python标准库中专门处理URL的模块，不需要另外安装
# urlparse：把完整URL拆成协议、主机、端口、路径、查询字符串等部分
# parse_qs：把查询字符串解析成Python字典，qs是query string（查询字符串）的缩写
from urllib.parse import urlparse, parse_qs

url = "https://api.example.com:443/chats/12/messages?limit=10&order=desc"

# urlparse(url)：
#   第1个位置参数传给参数url，表示要解析的完整URL字符串
#   返回ParseResult对象，可以通过.scheme、.hostname、.port等属性读取结果
parsed = urlparse(url)

print("协议：", parsed.scheme)
print("主机：", parsed.hostname)
print("端口：", parsed.port)
print("路径：", parsed.path)

# parse_qs(parsed.query)：
#   第1个位置参数传给参数qs，内容是URL中?后面的查询字符串
#   parsed.query此时是"limit=10&order=desc"
#   返回字典；一个参数可能出现多次，所以字典中的值默认使用列表保存
query_params = parse_qs(parsed.query)
print("查询参数：", query_params)

## 4. HTTP请求里有什么？

先把一个请求想象成Python字典：

In [ ]:
request = {
    "method": "POST",
    "path": "/chats",
    "headers": {"Content-Type": "application/json"},
    "body": {"title": "Python学习"},
}

print(request["method"])
print(request["path"])
print(request["body"]["title"])

请求中的常见部分：

- **method**：客户端想进行什么操作；
- **path**：操作哪个资源；
- **headers**：请求的附加说明，例如数据格式和身份凭证；
- **body**：客户端提交的主要数据，并非每个请求都有。

HTTP方法的常见约定：

| 方法 | 常见用途 | 示例 |
|---|---|---|
| GET | 查询资源 | 查询聊天记录 |
| POST | 创建资源或发起处理 | 创建聊天、发送消息 |
| PUT | 完整替换资源 | 替换完整用户资料 |
| PATCH | 部分修改资源 | 只修改聊天标题 |
| DELETE | 删除资源 | 删除聊天 |

这些是通用约定，不是Python语法。真正允许哪些方法，由后端程序决定。

## 5. HTTP响应里有什么？

服务器返回的响应通常包含：

- **status_code**：数字形式的处理结果；
- **headers**：响应的附加说明；
- **body**：返回给客户端的实际数据。

常见状态码：

| 状态码 | 含义 | 常见场景 |
|---|---|---|
| 200 | 请求成功 | 查询或修改成功 |
| 201 | 创建成功 | 成功创建聊天 |
| 204 | 成功但没有响应体 | 删除成功 |
| 400 | 请求本身不合理 | JSON格式或业务参数错误 |
| 401 | 尚未通过身份验证 | 没有登录或Token无效 |
| 403 | 身份已确认但没有权限 | 普通用户访问管理员接口 |
| 404 | 资源不存在 | 聊天ID不存在 |
| 422 | 提交的数据未通过验证 | FastAPI中字段类型不正确 |
| 500 | 服务器内部错误 | 后端程序出现未处理异常 |

`404`不是“Python报错”，而是服务器用HTTP规则告诉客户端：请求的资源不存在。

In [ ]:
response = {
    "status_code": 201,
    "headers": {"Content-Type": "application/json"},
    "body": {"id": 1, "title": "Python学习"},
}

print(response)

## 6. JSON是什么？

JSON是一种跨语言传输数据的文本格式。它看起来很像Python字典，但两者并不完全相同。

```json
{
  "message": "你好",
  "stream": false,
  "temperature": 0.7,
  "tools": null
}
```

主要区别：JSON使用`true`、`false`、`null`，Python使用`True`、`False`、`None`。网络上传输的是JSON文本，Python收到后通常会把它解析成字典或列表。

In [ ]:
# json是Python标准库中处理JSON数据的模块，不需要另外安装
import json

python_data = {"message": "你好", "stream": False, "tools": None}

# json.dumps(obj, ensure_ascii=False)：把Python对象转换成JSON字符串
#   obj参数：要转换的Python对象，这里传入python_data字典
#   ensure_ascii参数：是否强制把非ASCII字符转义；False表示直接保留中文
#   返回值是str。dumps可以记成dump string，即“转换成JSON字符串”
json_text = json.dumps(python_data, ensure_ascii=False)

# json.loads(s)：把JSON字符串解析回Python对象
#   s参数：待解析的JSON字符串，这里传入json_text
#   此处最外层JSON是对象，因此返回Python字典
#   loads可以记成load string，即“读取JSON字符串”
restored_data = json.loads(json_text)

print(json_text)
print(type(json_text))
print(restored_data)
print(type(restored_data))

## 7. API是什么？

API可以理解为程序对外提供的操作入口。一个API说明通常会告诉你：

- 使用什么请求方法；
- 访问什么路径；
- 需要提交哪些参数；
- 成功会返回什么；
- 失败可能返回什么。

例如：

```text
POST /messages
请求体：{"prompt": "什么是RAG？"}
成功响应：200 {"answer": "..."}
失败响应：400 {"detail": "prompt不能为空"}
```

API不是某个特定框架。FastAPI只是我们用来实现HTTP API的Python框架。

## 8. 路由是什么？

后端需要根据“方法 + 路径”决定调用哪段代码，这个匹配过程称为路由。

```text
GET  /health    → 执行健康检查函数
GET  /chats     → 执行查询聊天列表函数
POST /chats     → 执行创建聊天函数
```

`GET /chats`与`POST /chats`路径相同，但方法不同，因此是两个不同的接口。

In [ ]:
def simple_router(method: str, path: str) -> str:
    if method == "GET" and path == "/health":
        return "执行健康检查"
    if method == "GET" and path == "/chats":
        return "查询聊天列表"
    if method == "POST" and path == "/chats":
        return "创建聊天"
    return "没有匹配的接口"

print(simple_router("GET", "/health"))
print(simple_router("POST", "/chats"))
print(simple_router("DELETE", "/unknown"))

## 9. 本课作业

打开同目录的`http_api_exercise.py`并完成三个函数：

1. `build_response()`：构造统一的响应字典；
2. `create_chat()`：验证请求体并创建模拟聊天；
3. `route_request()`：根据HTTP方法和路径分发请求。

完成后，在本课目录运行：

```powershell
pytest -v
```

先自己尝试，不要修改测试文件来绕过测试。

## 10. 理解题（请把答案写在每道题下面）

1. 客户端与服务器分别负责什么？浏览器是不是唯一的客户端？

2. HTTP请求和HTTP响应分别由谁发送？

3. `GET /chats`和`POST /chats`为什么可以是两个不同的接口？

4. 路径参数与查询参数有什么区别？请各举一个例子。

5. Python字典和JSON完全相同吗？`None`转换成JSON后是什么？

6. `400`、`404`和`500`分别更可能表示谁或哪一部分出现了问题？

7. 如果创建聊天成功，为什么使用`201`通常比`200`表达得更准确？